# RAG evaluation with the Ragas Python SDK (modern metrics API)

This notebook shows a minimal end-to-end flow using Ragas modern metrics: build a small evaluation table (questions, retrieved contexts, and model answers), run metrics from `ragas.metrics.collections`, and inspect per-row and aggregate scores.

**Requirements**

- Python 3.10+ recommended.
- OpenAI-compatible API access: set `EVAL_LLM_API_KEY` and, when needed, `EVAL_LLM_BASE_URL` (the notebook also accepts `OPENAI_API_KEY` / `OPENAI_BASE_URL` as fallback).
- Optional separate embedding endpoint credentials: `EVAL_EMBED_API_KEY` and `EVAL_EMBED_BASE_URL`.
- Explicit model configuration in the notebook (`EVAL_LLM_MODEL` and optional `EVAL_EMBED_MODEL`; default is `text-embedding-3-small`).
- For retrieval-related metrics, use the same embedding model as the production RAG retriever whenever possible.
- Ragas calls the LLM (and embeddings where needed) multiple times; expect latency and cost proportional to dataset size and metrics.
- A dedicated virtual environment or workbench image reduces dependency conflicts with other projects.

**References**

- [Ragas documentation](https://docs.ragas.io/)
- Alauda AI docs: *Evaluating RAG with Ragas* — grouped metric overview and prerequisites (when browsing the documentation site)

## Install dependencies

Run this once per environment (for example a new workbench or virtualenv).

In [ ]:
# Use current kernel's Python so PATH does not point to another env
# If download is slow, add: -i https://pypi.tuna.tsinghua.edu.cn/simple
import sys
!{sys.executable} -m pip install "ragas" "datasets" "openai"

## Configure API credentials

Set `EVAL_LLM_API_KEY` (recommended) or `OPENAI_API_KEY` before running evaluation. If the endpoint is not the provider default, set `EVAL_LLM_BASE_URL` (or `OPENAI_BASE_URL`) as well.

Do not commit secrets into version control; use platform secret injection or notebook environment variables instead.

Optional: disable Ragas analytics (`RAGAS_DO_NOT_TRACK=true`) if required by policy.

In [ ]:
import os

# Config LLM API
# os.environ["EVAL_LLM_API_KEY"] = "sk-..."
# os.environ["EVAL_LLM_BASE_URL"] = "https://your-openai-compatible-endpoint/v1"  # optional
# os.environ["EVAL_LLM_MODEL"] = "..."

# Config Embeddings API
# os.environ["EVAL_EMBED_API_KEY"] = "sk-..."
# os.environ["EVAL_EMBED_BASE_URL"] = "https://your-embedding-endpoint/v1"
# os.environ["EVAL_EMBED_MODEL"] = "..."


# Optional: disable Ragas analytics
# os.environ["RAGAS_DO_NOT_TRACK"] = "true"

In [ ]:
from openai import AsyncOpenAI
from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory

EVAL_LLM_API_KEY = os.getenv("EVAL_LLM_API_KEY", os.getenv("OPENAI_API_KEY", ""))
EVAL_LLM_BASE_URL = os.getenv("EVAL_LLM_BASE_URL", os.getenv("OPENAI_BASE_URL", ""))
EVAL_LLM_MODEL = os.getenv("EVAL_LLM_MODEL", "")

EVAL_EMBED_API_KEY = os.getenv("EVAL_EMBED_API_KEY", EVAL_LLM_API_KEY)
EVAL_EMBED_BASE_URL = os.getenv("EVAL_EMBED_BASE_URL", EVAL_LLM_BASE_URL)
EVAL_EMBED_MODEL = os.getenv("EVAL_EMBED_MODEL", "")

if not EVAL_LLM_MODEL:
    raise RuntimeError("Set EVAL_LLM_MODEL to an available model ID from your endpoint.")

if not EVAL_EMBED_MODEL:
    raise RuntimeError("Set EVAL_EMBED_MODEL to an available model ID from your endpoint.")

llm_client = AsyncOpenAI(
    api_key=EVAL_LLM_API_KEY,
    base_url=EVAL_LLM_BASE_URL or None,
)
embed_client = AsyncOpenAI(
    api_key=EVAL_EMBED_API_KEY,
    base_url=EVAL_EMBED_BASE_URL or None,
)

llm = llm_factory(EVAL_LLM_MODEL, client=llm_client)
embeddings = OpenAIEmbeddings(
    model=EVAL_EMBED_MODEL,
    client=embed_client,
)

print(f"llm_base_url={EVAL_LLM_BASE_URL or '(provider default)'}")
print(f"llm={EVAL_LLM_MODEL}")
print(f"embed_base_url={EVAL_EMBED_BASE_URL or '(provider default)'}")
print(f"embeddings={EVAL_EMBED_MODEL}")

## Build an evaluation dataset

For the modern metrics API in this notebook, organize data as row samples (one dictionary per sample).

Each row uses argument-aligned names:

- `user_input`: user query
- `retrieved_contexts`: list of retrieved passages for that row
- `response`: model response to score
- `reference`: reference answer or expected facts (needed by retrieval/reference-based metrics)

This row-first structure matches `ascore()` usage and avoids extra mapping.

In [ ]:
from datasets import Dataset

samples = [
    {
        "user_input": "What is the capital of France?",
        "retrieved_contexts": [
            "Paris is the capital and most populous city of France."
        ],
        "response": "The capital of France is Paris.",
        "reference": "Paris",
    },
    {
        "user_input": "Who patented an early practical telephone?",
        "retrieved_contexts": [
            "Alexander Graham Bell was a Scottish-born inventor who patented the first practical telephone."
        ],
        "response": "Alexander Graham Bell patented an early practical telephone.",
        "reference": "Alexander Graham Bell",
    },
    {
        "user_input": "What is photosynthesis?",
        "retrieved_contexts": [
            "Photosynthesis is the process by which plants convert light energy into chemical energy."
        ],
        "response": "Photosynthesis is how plants turn sunlight into chemical energy.",
        "reference": "Plants convert light energy into chemical energy during photosynthesis.",
    },
]

dataset = Dataset.from_list(samples)
dataset

## Run evaluation (modern metrics)

- **Faithfulness**: whether the answer is supported by the retrieved contexts.
- **Answer relevancy**: whether the answer addresses the question.

This section uses metrics from `ragas.metrics.collections` with the modern embeddings/LLM interfaces.

In [ ]:
from ragas.metrics.collections import AnswerRelevancy, Faithfulness

faithfulness_metric = Faithfulness(llm=llm)
answer_relevancy_metric = AnswerRelevancy(llm=llm, embeddings=embeddings)


async def score_baseline_rows(ds):
    rows = ds.to_list()
    scored = []
    for row in rows:
        faithfulness_result = await faithfulness_metric.ascore(
            user_input=row["user_input"],
            response=row["response"],
            retrieved_contexts=row["retrieved_contexts"],
        )
        answer_relevancy_result = await answer_relevancy_metric.ascore(
            user_input=row["user_input"],
            response=row["response"],
        )
        scored.append(
            {
                "user_input": row["user_input"],
                "faithfulness": faithfulness_result.value,
                "answer_relevancy": answer_relevancy_result.value,
            }
        )
    return scored


baseline_scores = await score_baseline_rows(dataset)
faithfulness_avg = sum(item["faithfulness"] for item in baseline_scores) / len(baseline_scores)
answer_relevancy_avg = sum(item["answer_relevancy"] for item in baseline_scores) / len(baseline_scores)

print("Aggregate means:")
print(f"faithfulness={faithfulness_avg:.4f}")
print(f"answer_relevancy={answer_relevancy_avg:.4f}")
print("\nPer-row scores:")
for idx, item in enumerate(baseline_scores, start=1):
    print(
        f"{idx}. user_input={item['user_input']} | "
        f"faithfulness={item['faithfulness']:.4f} | "
        f"answer_relevancy={item['answer_relevancy']:.4f}"
    )

## Add retrieval-focused metrics (modern metrics)

- **Context precision**: whether retrieved chunks are useful for answering the question.
- **Context recall**: whether retrieved contexts cover what the reference (`ground_truth`) states.

This pass issues additional LLM calls. If validation errors mention missing columns, adjust the dataset or choose another metric variant per the [Ragas metrics documentation](https://docs.ragas.io/).

In [ ]:
from ragas.metrics.collections import ContextPrecision, ContextRecall

context_precision_metric = ContextPrecision(llm=llm)
context_recall_metric = ContextRecall(llm=llm)


async def score_retrieval_rows(ds):
    rows = ds.to_list()
    scored = []
    for row in rows:
        context_precision_result = await context_precision_metric.ascore(
            user_input=row["user_input"],
            reference=row["reference"],
            retrieved_contexts=row["retrieved_contexts"],
        )
        context_recall_result = await context_recall_metric.ascore(
            user_input=row["user_input"],
            retrieved_contexts=row["retrieved_contexts"],
            reference=row["reference"],
        )
        scored.append(
            {
                "user_input": row["user_input"],
                "context_precision": context_precision_result.value,
                "context_recall": context_recall_result.value,
            }
        )
    return scored


retrieval_scores = await score_retrieval_rows(dataset)
context_precision_avg = sum(item["context_precision"] for item in retrieval_scores) / len(retrieval_scores)
context_recall_avg = sum(item["context_recall"] for item in retrieval_scores) / len(retrieval_scores)

print("Aggregate means:")
print(f"context_precision={context_precision_avg:.4f}")
print(f"context_recall={context_recall_avg:.4f}")
print("\nPer-row scores:")
for idx, item in enumerate(retrieval_scores, start=1):
    print(
        f"{idx}. user_input={item['user_input']} | "
        f"context_precision={item['context_precision']:.4f} | "
        f"context_recall={item['context_recall']:.4f}"
    )

## Troubleshooting

- **Model not found (`Model Not Exist`)**: set `EVAL_LLM_MODEL` (and, when overridden, `EVAL_EMBED_MODEL`) to an available model ID from the endpoint (for example via `/models`).
- **Credentials or endpoint setup**: set `EVAL_LLM_API_KEY` / `EVAL_LLM_BASE_URL` (fallback: `OPENAI_API_KEY` / `OPENAI_BASE_URL`). If embeddings use a separate endpoint, also set `EVAL_EMBED_API_KEY` / `EVAL_EMBED_BASE_URL`.
- **Notebook async execution**: this notebook uses `await metric.ascore(...)` in cells. If running outside notebook contexts, use `asyncio.run(...)` or metric `.score(...)` in synchronous scripts.
- **Version-related warnings**: metric classes and signatures can change across Ragas releases. Pin package versions for reproducible runs and confirm behavior against [docs.ragas.io](https://docs.ragas.io/).